# Questão 2 - Gestão do consumo de energia

Integrantes: Eduardo - RM561474; João Abe - RM561446. Grupo 4. Seed 4. A geração usa `config.py` e quatro regiões por hora. O índice `n` dos experimentos significa horas agregadas, não linhas do CSV.


In [ ]:
from pathlib import Path
import sys
import json

RAIZ = Path.cwd().resolve()
if not (RAIZ / 'src').is_dir():
    RAIZ = RAIZ.parent
if not (RAIZ / 'src').is_dir():
    raise RuntimeError('Abra o notebook dentro da pasta checkpoint4/notebooks.')
if str(RAIZ) not in sys.path:
    sys.path.insert(0, str(RAIZ))
from config import SEED, REPETICOES
print('Grupo e seed:', SEED)


Grupo e seed: 4


## Gerar e medir

Esta célula gera 20.000 registros, agrega 5.000 horas e mede cada algoritmo para seis tamanhos. Os tempos usam três repetições por padrão. A memória é medida em uma execução separada; essa etapa pode levar mais tempo.


In [ ]:
from src.energia import executar_questao2
resultado = executar_questao2(RAIZ, seed=SEED, repeticoes=REPETICOES)
print('Registros:', resultado['registros_brutos'])
print('Horas:', resultado['horas_agregadas'])
print('Intervalo crítico:', resultado['intervalo'])
for metodo in ['forca_bruta', 'dividir_conquistar']:
    print(metodo, resultado[metodo])
assert all(resultado['forca_bruta'][k] == resultado['dividir_conquistar'][k]
           for k in ['inicio', 'fim', 'soma'])


Registros: 20000
Horas: 5000
Intervalo crítico: {'inicio': '2026-05-14T08:00', 'fim': '2026-05-26T19:00', 'duracao_horas': 300}
forca_bruta {'inicio': 3200, 'fim': 3499, 'soma': 38328640, 'operacoes': 12502500}
dividir_conquistar {'inicio': 3200, 'fim': 3499, 'soma': 38328640, 'operacoes': 61808}


## Criticidade e consultas

Em cada registro, calculamos `(consumo - capacidade_disponivel) * prioridade * custo`. O custo está em centavos por kWh. Depois somamos os valores das regiões de cada horário. A pontuação é um índice didático; não é uma conta de luz nem uma regra oficial do sistema elétrico.

Sobras geram valores negativos e excessos geram valores positivos. O melhor intervalo deve ter pelo menos uma hora. Em empate, usamos menor início e depois menor fim.


In [ ]:
print('Horas positivas:', resultado['horas_positivas'])
print('Horas negativas:', resultado['horas_negativas'])
print('Consumo acumulado por região:')
print(json.dumps(resultado['consultas']['consumo_acumulado_por_regiao_kwh'], ensure_ascii=False, indent=2))
print('Maiores picos:')
for pico in resultado['consultas']['cinco_maiores_picos']:
    print(pico)


Horas positivas: 522
Horas negativas: 4478
Consumo acumulado por região:
{
  "Nordeste": 5204035,
  "Norte": 4201771,
  "Sudeste": 7702921,
  "Sul": 4705379
}
Maiores picos:
{'timestamp': '2026-05-16T14:00', 'consumo': 5724}
{'timestamp': '2026-05-19T13:00', 'consumo': 5723}
{'timestamp': '2026-05-19T12:00', 'consumo': 5722}
{'timestamp': '2026-05-26T14:00', 'consumo': 5716}
{'timestamp': '2026-05-21T14:00', 'consumo': 5715}


![Consumo e intervalo crítico](../figures/questao2/01_serie_temporal.png)

## Força bruta e dividir e conquistar

Na força bruta, fixamos cada início e avançamos o final, atualizando uma soma acumulada. São `n(n+1)/2` intervalos.

Em dividir e conquistar, o caso-base contém um único valor. Dividimos pelos índices, resolvemos esquerda e direita e calculamos o cruzamento: melhor sufixo da esquerda mais melhor prefixo da direita. Comparamos as três respostas usando o mesmo desempate.

![Três níveis da decomposição real](../figures/questao2/02_arvore_divisao.png)


In [ ]:
from src.brute_force import forca_bruta
from src.divide_conquer import dividir_conquistar
for valores in [[4, -6, 3, 5, -2], [-8, -2, -5], [0, 0]]:
    a = forca_bruta(valores)
    b = dividir_conquistar(valores)
    print(valores, '->', (a['inicio'], a['fim'], a['soma']))
    assert all(a[k] == b[k] for k in ['inicio', 'fim', 'soma'])


[4, -6, 3, 5, -2] -> (2, 3, 8)
[-8, -2, -5] -> (1, 1, -2)
[0, 0] -> (0, 0, 0)


## Escalabilidade
Os tempos variam com a máquina. As contagens de operações são determinísticas e contam as adições nas varreduras.

In [ ]:
print('n horas | algoritmo | tempo médio (ms) | somas | pico extra (bytes)')
for linha in resultado['escalabilidade']:
    print(linha['n_horas'], '|', linha['algoritmo'], '|',
          round(linha['tempo_medio_s'] * 1000, 4), '|',
          linha['operacoes'], '|', linha['pico_memoria_bytes'])


n horas | algoritmo | tempo médio (ms) | somas | pico extra (bytes)
100 | Força bruta | 0.1526 | 5050 | 472
100 | Dividir e conquistar | 0.0705 | 672 | 824
250 | Força bruta | 0.9948 | 31375 | 472
250 | Dividir e conquistar | 0.1901 | 1994 | 920
500 | Força bruta | 4.8058 | 125250 | 472
500 | Dividir e conquistar | 0.4244 | 4488 | 1872
1000 | Força bruta | 19.0398 | 500500 | 472
1000 | Dividir e conquistar | 0.9988 | 9976 | 2192
2000 | Força bruta | 71.5461 | 2001000 | 492
2000 | Dividir e conquistar | 2.0113 | 21952 | 2448
5000 | Força bruta | 454.5899 | 12502500 | 492
5000 | Dividir e conquistar | 6.4896 | 61808 | 2960


![Tempo por tamanho da entrada](../figures/questao2/03_escalabilidade.png)

## Conclusão

Os dois algoritmos encontram o mesmo resultado exato. A força bruta usa tempo `O(n²)` e espaço auxiliar `O(1)`. Dividir e conquistar usa tempo `O(n log n)` e espaço auxiliar `O(log n)`, pois passa índices sem copiar a lista em cada chamada.

Com quatro regiões fixas, 1.000.000 de registros equivalem a 250.000 horas. A força bruta examinaria 31.250.125.000 intervalos. Dividir e conquistar continua sendo a alternativa mais viável entre as duas. Essa comparação é teórica; não fizemos uma medição com um milhão de registros.
